# Session 30: Hypothesis Testing

Data Science with Python, Level 2, Stage 4: Statistics

## Learning Objectives

By the end of this session you will be able to:

1. State a null and an alternative hypothesis.
2. Explain what a p-value measures.
3. Run a permutation test by simulation.
4. Use t-tests and chi-square tests in SciPy.
5. Tell statistical from practical significance.
6. Avoid the most common testing mistakes.

## Where We Are

In Session 29 you learned that experiments are the surest way to find causes. Companies run experiments all the time: they show half of their website visitors the old page (A) and half a new page (B), and compare. This is called an **A/B test**.

The new page will almost never score exactly the same as the old one. The question is whether the difference is **bigger than chance alone would produce**. Today we use a practice dataset of **4,000 visitors** in such an experiment and answer it with a hypothesis test.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(30)
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
ab = pd.read_csv(DATA_URL + "ab_test.csv")

## 1. The Logic of a Test

In [ ]:
summary = ab.groupby("group").agg(visitors=("visitor_id", "count"),
                                  conversion=("converted", "mean"),
                                  time_on_page=("time_on_page_s", "mean"))
summary.round(3)

## 2. A Permutation Test

If the page made **no** difference (H0), the labels A and B are meaningless: any visitor could have been in either group. So we **shuffle** the labels many times, and each time compute the difference in conversion. This shows the differences that chance alone produces. The p-value is the share of shuffles with a difference at least as big as the real one.

In [ ]:
converted = ab["converted"].to_numpy()
is_b = (ab["group"] == "B").to_numpy()
observed = converted[is_b].mean() - converted[~is_b].mean()

diffs = []
for _ in range(5000):
    shuffled = rng.permutation(is_b)
    diffs.append(converted[shuffled].mean() - converted[~shuffled].mean())
diffs = np.array(diffs)

print("Observed difference:", round(observed, 4))
print("p-value:", (np.abs(diffs) >= abs(observed)).mean())

In [ ]:
sns.histplot(diffs, bins=40)
plt.axvline(observed, color="tab:red", linewidth=2, label="observed")
plt.title("Differences produced by chance alone")
plt.xlabel("Difference in conversion rate")
plt.legend()
plt.show()

## 3. Tests in SciPy

In [ ]:
a = ab.loc[ab["group"] == "A", "time_on_page_s"]
b = ab.loc[ab["group"] == "B", "time_on_page_s"]

t, p = stats.ttest_ind(b, a, equal_var=False)
print(f"Mean A {a.mean():.1f} s, mean B {b.mean():.1f} s")
print(f"t = {t:.2f}, p = {p:.4f}")

In [ ]:
table = pd.crosstab(ab["group"], ab["converted"])
print(table)

chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"chi2 = {chi2:.1f}, p = {p:.2e}")

| Question | Data | SciPy |
| --- | --- | --- |
| Do two groups differ in their mean? | Numbers, two groups | `stats.ttest_ind(a, b, equal_var=False)` |
| Did the same people change? | Numbers, before and after | `stats.ttest_rel(before, after)` |
| Do several groups differ in their mean? | Numbers, three or more groups | `stats.f_oneway(a, b, c)` |
| Are two categories related? | Counts in a table | `stats.chi2_contingency(table)` |
| Any statistic, few assumptions | Anything | A permutation test |

## 4. Errors and Significance

In [ ]:
diff = b.mean() - a.mean()
pooled_sd = np.sqrt((a.var() + b.var()) / 2)
print(f"Difference: {diff:.1f} seconds")
print(f"Effect size (Cohen's d): {diff / pooled_sd:.2f}")

print("Conversion lift:", round((summary.loc["B", "conversion"] / summary.loc["A", "conversion"] - 1) * 100, 1), "percent")

In [ ]:
p_values = []
for _ in range(1000):
    x = rng.normal(0, 1, 30)
    y = rng.normal(0, 1, 30)
    p_values.append(stats.ttest_ind(x, y).pvalue)

print("Share of false alarms:", np.mean(np.array(p_values) < 0.05))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Misread p | Saying p is the chance that H0 is true | p is the chance of data this extreme if H0 were true. |
| Small but significant | Treating every significant result as important | Report the size of the effect, not only p. |
| Fishing | Running many tests and reporting only significant ones | Plan the tests in advance; report them all. |
| No difference proven | Concluding no effect because p > 0.05 | Say the data does not show an effect; it may be too small. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Devices

Does conversion differ between mobile and desktop visitors? Build the crosstab and run a chi-square test.

In [ ]:
# Write your code below

### In-Class Exercise 2: Pages Viewed

Test whether visitors in group B view more pages than those in group A with a t-test. Report the means, the p-value and your decision at alpha = 0.05.

In [ ]:
# Write your code below

### In-Class Exercise 3: Permutation on Time

Write a permutation test for the difference in mean time on page between B and A with 3,000 shuffles, and compare the p-value with the t-test from the lecture.

In [ ]:
# Write your code below

### In-Class Exercise 4: Sample Size Matters

Take only the first 200 visitors of the dataset and run the chi-square test on group and conversion again. Is the result still significant? Why might it change?

In [ ]:
# Write your code below

## Summary

- H0 says nothing is going on; the test asks how surprising the data is.
- The p-value: the chance of data this extreme if H0 were true.
- Permutation tests shuffle labels to show what chance produces.
- `ttest_ind` compares means; `chi2_contingency` compares categories.
- Significant is not the same as important: report effect sizes.
- Many tests produce false alarms: plan tests in advance.

## What's Next

Stage 4 is complete. Stage 5 begins machine learning: next session you will fit your first model, a straight line that predicts apartment prices, with scikit-learn.